# Power up Data
<hr>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import requests
import seaborn as sns
from sklearn.linear_model import LinearRegression

## Loading the Power up data

In [2]:
powerup_df = pd.read_csv("Data/powerup-data.csv", 
                         parse_dates=["Date"], 
                         header=None, 
                         names=["Date","Powerup","Time Spawned","Time Activated","Time Effect Ended"], 
                         dtype={"Powerup":"string", "Time Spawned":"float64", "Time Activated":"float64","Time Effect Ended":"float64"})

powerup_df = powerup_df.replace(r"^\s*$", pd.NA, regex=True)

powerup_df.set_index("Date", inplace=True)

## What data is stored and why
<hr>

### Date

The Date is the id for which game the powerup was created in. All records with the same Date happened during the same game

### Powerup

This is the name of the powerup

### Time Spawned

This is how many seconds into the game the powerup was spawned

### Time Activated

This is what time (if any) into the game the power up was activated

### Time Effect Ended

This is what time (if any) into the game the powerups effect ended


Checking the data is here

In [3]:
print(len(powerup_df))

powerup_df.head()

31


,Powerup,Time Spawned,Time Activated,Time Effect Ended
Date,,,,
2026-08-20 19:53:15.689284,Eat_Enemy_Powerup,22.549473,25.925230,30.938932
2026-08-20 19:53:15.689284,Eat_Enemy_Powerup,45.070239,45.870412,50.875540
2026-08-20 19:53:15.689284,Score_Increase_Powerup,15.043289,15.925274,20.856675
2026-08-20 19:53:15.689284,Score_Increase_Powerup,30.052839,30.883667,35.842999
2026-08-20 19:53:15.689284,Temp_Change_Colour_Powerup,7.528527,NaN,NaN


# Handling Empty Values
<hr>

All records missing a `Date`, `Powerup` or `Time Spawned` are removed

If the `Time Activated` is NaN that means the effect was never activated

If the `Time Effect Ended` is NaN that means the effect never ended. However if a record has a `Time Effect Ended` but not a `Time Activated` then the record will be removed

In [4]:
powerup_df = powerup_df[powerup_df.notna()]

powerup_df.dropna(subset=["Powerup","Time Spawned"], inplace=True)

powerup_df = powerup_df[~(powerup_df["Time Effect Ended"].notna() & powerup_df["Time Activated"].isna())]
  

## Adding game-results data
<hr>

In [5]:
gr_df = pd.read_csv("Data/game-results.csv", parse_dates=[0,1], header=None, names=["Game Version", "Date", "Time Survived", "Score"], dtype={"Time Survived":"float64","Score":"Int64"})

gr_df = gr_df.replace(r"^\s*$", pd.NA, regex=True)
gr_df["Date"] = gr_df["Date"].str.strip()
gr_df["Date"] = pd.to_datetime(gr_df["Date"])

gr_df.set_index("Date", inplace=True)

In [6]:
gr_df = gr_df[gr_df.index.notna()]

gr_df.dropna(subset=["Time Survived", "Score"], inplace=True)

missing = gr_df["Game Version"].isna()

for idx in gr_df.index[missing]:
    pos = gr_df.index.get_loc(idx)

    # First or Last
    if pos == 0 or pos == len(gr_df) -1:
        df.drop(idx, inplace=True)
        continue

    above = gr_df.iloc[pos -1]["Game Version"]
    below = gr_df.iloc[pos + 1]["Game Version"]

    if pd.notna(above) and pd.notna(below) and above == below:
        gr_df.at[idx, "Game Version"] = above
    else:
        gr_df.drop(idx, inplace=True)

In [7]:
gr_df.head()

,Game Version,Time Survived,Score
Date,,,
2026-08-11 22:02:53.976116,2026-08-10 23:55:20.054130,10.757392,260
2026-08-11 22:03:49.405594,2026-08-10 23:55:20.054130,24.408522,933
2026-08-11 22:07:52.788656,2026-08-10 23:55:20.054130,1.700332,31
2026-08-11 22:08:16.921228,2026-08-10 23:55:20.054130,12.781978,340
2026-08-11 22:09:49.791919,2026-08-10 23:55:20.054130,0.995767,17


## Average Time Survived After Powerup Activated
<hr>

In [16]:
com_df = powerup_df.merge(gr_df, left_index=True, right_index=True, how="inner")
com_df["Time Survived After Activated"] = (com_df["Time Survived"]-com_df["Time Activated"])

In [17]:
average_times = com_df.groupby("Powerup")["Time Survived After Activated"].mean()

print(average_times)

Powerup
Eat_Enemy_Powerup             24.366189
Score_Increase_Powerup        28.728110
Temp_Change_Colour_Powerup    23.150288
Name: Time Survived After Activated, dtype: float64


## Average Time Between Spawn and Activation
<hr>
<br>

### Why do this?

By calculating the average `Time Between Spawn And Activation` them we can see how much a player prioritises getting them.

Calculating a version of the average which adds unactivated Powerups as being activated at the game end and comparing the difference between it and the orignal should help us work out if players are avoiding a powerup altogether.

If the change is `0` that means every time that powerup has spawned it has been activated by the player<br>
If the change is `<0` that means for unactived powerups the player is dying before they would normally activate them. Possibly because they view getting them as very important <br>
If the change is `>0` that means players are deliberatly avoiding taking the power up. The greater the difference the more they are avoiding

In [18]:
com_df["Time Between Spawn And Activation"] = (com_df["Time Activated"]-com_df["Time Spawned"])

In [21]:
def Time_Between_Spawn_Activation():
    temp_df = com_df.copy()
    temp_df["Time Activated"] = temp_df["Time Activated"].fillna(temp_df["Time Survived"])
    temp_df["Time Between Spawn And Activation"] = (temp_df["Time Activated"]-temp_df["Time Spawned"])
    average_nofill = com_df.groupby("Powerup")["Time Between Spawn And Activation"].mean()
    average_fill = temp_df.groupby("Powerup")["Time Between Spawn And Activation"].mean()

    print(f"Average Time between Spawn and Activation no fill: {average_nofill}")
    print(f"Average Time between Spawn and Activation with fill: {average_fill}")
    difference = average_fill-average_nofill
    print(f"Change between fill and no fill: {difference}")

In [22]:
Time_Between_Spawn_Activation()

Average Time between Spawn and Activation no fill: Powerup
Eat_Enemy_Powerup             3.150060
Score_Increase_Powerup        2.071129
Temp_Change_Colour_Powerup    2.076246
Name: Time Between Spawn And Activation, dtype: float64
Average Time between Spawn and Activation with fill: Powerup
Eat_Enemy_Powerup             3.150060
Score_Increase_Powerup        2.071129
Temp_Change_Colour_Powerup    4.862853
Name: Time Between Spawn And Activation, dtype: float64
Change between fill and no fill: Powerup
Eat_Enemy_Powerup             0.000000
Score_Increase_Powerup        0.000000
Temp_Change_Colour_Powerup    2.786607
Name: Time Between Spawn And Activation, dtype: float64
